<a href="https://colab.research.google.com/github/thatcherty/ML-Algo-Selection/blob/main/ML_Algo_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Algo Selection

In [35]:
# Common
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from pandas.errors import ParserError

# requires pip install ucimlrepo
from ucimlrepo import fetch_ucirepo


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score, accuracy_score

# additional for Neural Network
# include MLP or CNN
import tensorflow as tf
from tensorflow import keras

# Helper functions
def build_preprocessor(X: pd.DataFrame):
    # Detect column types
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [col for col in X.columns if col not in numeric_cols]

    # Numeric preprocessing
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Categorical preprocessing
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Combine both
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

    return preprocessor

dataset = fetch_ucirepo(id=1)
print(dataset.metadata)


{'uci_id': 1, 'name': 'Abalone', 'repository_url': 'https://archive.ics.uci.edu/dataset/1/abalone', 'data_url': 'https://archive.ics.uci.edu/static/public/1/data.csv', 'abstract': 'Predict the age of abalone from physical measurements', 'area': 'Biology', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Tabular'], 'num_instances': 4177, 'num_features': 8, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': [], 'target_col': ['Rings'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1994, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C55C7W', 'creators': ['Warwick Nash', 'Tracy Sellers', 'Simon Talbot', 'Andrew Cawthorn', 'Wes Ford'], 'intro_paper': None, 'additional_info': {'summary': 'Predicting the age of abalone from physical measurements.  The age of abalone is determined by cutting the shell through the cone, staining it, and counting the number of rings through a microscope -- 

In [36]:
import argparse
import re
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urljoin, urlparse, urlunparse

import requests
from bs4 import BeautifulSoup


BASE_URL = (
    "https://archive.ics.uci.edu/datasets"
    "?Task=Classification"
    "&Types=Multivariate"
    "&Types=Tabular"
    "&Python=true"
    "&skip=0"
    "&take=10"
    "&sort=desc"
    "&orderBy=NumHits"
    "&search="
)

USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0 Safari/537.36"
)


def build_page_url(base_url: str, *, skip: int, take: int) -> str:
    parsed = urlparse(base_url)

    # Keep repeated query params like Types=...
    params = parse_qsl(parsed.query, keep_blank_values=True)

    new_params = []
    skip_set = False
    take_set = False

    for key, value in params:
        if key == "skip":
            new_params.append(("skip", str(skip)))
            skip_set = True
        elif key == "take":
            new_params.append(("take", str(take)))
            take_set = True
        else:
            new_params.append((key, value))

    if not skip_set:
        new_params.append(("skip", str(skip)))
    if not take_set:
        new_params.append(("take", str(take)))

    new_query = urlencode(new_params, doseq=True)
    return urlunparse(parsed._replace(query=new_query))


def extract_total_count(page_text: str) -> int | None:
    match = re.search(r"\b\d+\s+to\s+\d+\s+of\s+(\d+)\b", page_text)
    return int(match.group(1)) if match else None


def parse_dataset_links(html: str, page_url: str) -> list[int]:
    soup = BeautifulSoup(html, "html.parser")

    anchors = soup.select("a[href*='/dataset/']")
    dataset_ids = []
    seen = set()

    for a in anchors:
        href = (a.get("href") or "").strip()
        if not href:
            continue

        full_url = urljoin(page_url, href)

        match = re.search(r"/dataset/(\d+)", full_url)
        if not match:
            continue

        dataset_id = int(match.group(1))

        if dataset_id in seen:
            continue
        seen.add(dataset_id)
        dataset_ids.append(dataset_id)

    return dataset_ids


def scrape_all_datasets(base_url: str, *, take: int = 25, timeout: int = 30) -> list[int]:
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    all_ids = []
    seen_ids = set()
    skip = 0
    total_count = None

    while True:
        page_url = build_page_url(base_url, skip=skip, take=take)

        response = session.get(page_url, timeout=timeout)
        response.raise_for_status()

        html = response.text
        text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)

        if total_count is None:
            total_count = extract_total_count(text)
            print(f"Total count reported: {total_count}")

        page_ids = parse_dataset_links(html, page_url)
        new_ids = [dataset_id for dataset_id in page_ids if dataset_id not in seen_ids]

        if not new_ids:
            break

        for dataset_id in new_ids:
            seen_ids.add(dataset_id)
            all_ids.append(dataset_id)

        if total_count is not None and len(all_ids) >= total_count:
            break

        if len(page_ids) < take:
            break

        skip += take

    return all_ids


def get_data() -> list[int]:
    parser = argparse.ArgumentParser(
        description="Scrape UCI dataset IDs matching the filtered URL."
    )
    parser.add_argument("--url", default=BASE_URL, help="Filtered UCI listing URL to scrape.")
    parser.add_argument(
        "--take",
        type=int,
        default=25,
        help="Rows requested per page while paginating."
    )
    parser.add_argument(
        "--timeout",
        type=int,
        default=30,
        help="HTTP timeout in seconds."
    )
    args = parser.parse_args([])

    ids = scrape_all_datasets(args.url, take=args.take, timeout=args.timeout)

    if not ids:
        raise SystemExit("No dataset IDs were found. The page structure may have changed.")

    return ids

In [37]:

# extract datasets

exclude_types: list[str] = ["Time-Series", "Image", "Sequential", "Spatiotemporal", "Text", "Other"]

ids = get_data()
print(ids)

datasets = []
excluded_ids = []

for dataset_id in ids:
    try:
        dataset = fetch_ucirepo(id=dataset_id)
    except ParserError as e:
        print(f"Skipping {dataset_id} due to parse error: {e}")
        excluded_ids.append(dataset_id)
        continue
    except Exception as e:
        print(f"Skipping {dataset_id} due to fetch error: {e}")
        excluded_ids.append(dataset_id)
        continue

    characteristics = dataset.metadata.get("characteristics") or []

    if any(exclude in characteristics for exclude in exclude_types):
        print(f"Skipping {dataset_id} due to characteristics: {characteristics}")
        excluded_ids.append(dataset_id)
        continue

    datasets.append(dataset)

print("Loaded dataset count:", len(datasets))
print("Failed dataset count:", len(excluded_ids))



Total count reported: 167
[53, 45, 186, 17, 222, 2, 320, 352, 109, 19, 697, 350, 144, 73, 544, 1, 891, 94, 468, 159, 15, 296, 14, 519, 602, 27, 20, 336, 242, 601, 292, 174, 327, 80, 42, 545, 267, 856, 31, 111, 555, 967, 62, 332, 597, 863, 373, 59, 529, 579, 936, 942, 46, 848, 563, 890, 52, 878, 383, 225, 572, 850, 857, 887, 445, 158, 151, 101, 571, 145, 938, 915, 484, 880, 864, 365, 110, 547, 759, 193, 244, 732, 763, 33, 105, 76, 143, 451, 176, 603, 198, 12, 16, 264, 379, 471, 760, 212, 39, 461, 172, 467, 90, 536, 50, 503, 47, 419, 161, 117, 81, 357, 911, 485, 58, 342, 537, 827, 30, 329, 582, 799, 247, 40, 43, 26, 565, 149, 146, 13, 270, 38, 372, 148, 913, 367, 257, 277, 713, 300, 54, 28, 728, 83, 722, 22, 567, 184, 78, 107, 95, 63, 755, 3, 70, 44, 75, 91, 23, 74, 32, 96, 82, 18, 88, 8, 147]
Skipping 352 due to characteristics: ['Multivariate', 'Sequential', 'Time-Series']


/usr/local/lib/python3.12/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Skipping 601 due to characteristics: ['Multivariate', 'Time-Series']


/usr/local/lib/python3.12/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (0,5,6,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Skipping 597 due to characteristics: ['Multivariate', 'Time-Series']
Skipping 942 due to characteristics: ['Tabular', 'Sequential', 'Multivariate']
Skipping 445 due to characteristics: ['Multivariate', 'Time-Series']
Skipping 938 due to characteristics: ['Tabular', 'Image']
Skipping 484 due to characteristics: ['Multivariate', 'Text']
Skipping 864 due to characteristics: ['Multivariate', 'Time-Series']
Skipping 763 due to characteristics: ['Tabular', 'Multivariate', 'Other']
Skipping 264 due to characteristics: ['Multivariate', 'Sequential', 'Time-Series']
Skipping 760 due to characteristics: ['Sequential', 'Multivariate', 'Time-Series']
Skipping 461 due to characteristics: ['Multivariate', 'Text']
Skipping 172 due to characteristics: ['Multivariate', 'Sequential', 'Time-Series']
Skipping 536 due to characteristics: ['Multivariate', 'Sequential', 'Time-Series']
Skipping 357 due to characteristics: ['Multivariate', 'Time-Series']
Skipping 911 due to characteristics: ['Tabular', 'Other']

In [41]:
print(excluded_ids)

datasets[0].metadata

[352, 601, 597, 942, 445, 938, 484, 864, 763, 264, 760, 461, 172, 536, 357, 911, 485, 247, 270, 38, 755]


{'uci_id': 53,
 'name': 'Iris',
 'repository_url': 'https://archive.ics.uci.edu/dataset/53/iris',
 'data_url': 'https://archive.ics.uci.edu/static/public/53/data.csv',
 'abstract': 'A small classic dataset from Fisher, 1936. One of the earliest known datasets used for evaluating classification methods.\n',
 'area': 'Biology',
 'tasks': ['Classification'],
 'characteristics': ['Tabular'],
 'num_instances': 150,
 'num_features': 4,
 'feature_types': ['Real'],
 'demographics': [],
 'target_col': ['class'],
 'index_col': None,
 'has_missing_values': 'no',
 'missing_values_symbol': None,
 'year_of_dataset_creation': 1936,
 'last_updated': 'Tue Sep 12 2023',
 'dataset_doi': '10.24432/C56C76',
 'creators': ['R. A. Fisher'],
 'intro_paper': {'ID': 191,
  'type': 'NATIVE',
  'title': 'The Iris data set: In search of the source of virginica',
  'authors': 'A. Unwin, K. Kleinman',
  'venue': 'Significance, 2021',
  'year': 2021,
  'journal': 'Significance, 2021',
  'DOI': '1740-9713.01589',
  'UR

In [ ]:
# excluded dataset ID's [352, 601, 597, 942, 445, 938, 484, 864, 763, 264, 760, 461, 172, 536, 357, 911, 485, 247, 270, 38, 755]

count = 0

test_log = 0
test_nn = 0
test_tree = 1

# Create final dataset obj
algo_selection = pd.DataFrame({"Dataset": [], "Feature Count": [], "Instances": [], "Missing Values": [], "Recommended Algorithm": []})

for dataset in datasets:
  if count == 25:
    break
  count += 1

  # Address multiple target variables
  target_y = dataset.data.targets

  # if no target, do not include in evaluation
  if target_y is None or target_y.empty:
    print(f"No target variables for {dataset.metadata["uci_id"]}")
    continue

  # if one target, simply use
  if len(target_y.columns) == 1:
    target_y = dataset.data.targets

  # if > 1 target, split to columns
  else:
    target_y = target_y.columns.tolist()

  # model assessment loop to account for multiple target var
  for target in target_y:
    X = dataset.data.features
    y = dataset.data.targets[target]

    # Get dataset features
    name = dataset.metadata['name']
    feature_count = dataset.metadata['num_features']
    instance_count = dataset.metadata['num_instances']
    missing_values = 'Yes' if 'Yes' in dataset.variables['missing_values'] else 'No'

    new_row = {'Dataset': name, 'Feature Count': feature_count, 'Instances': instance_count, 'Missing Values': missing_values}
    new_row_df = pd.DataFrame([new_row])

    # Add features to final dataset obj
    algo_selection = pd.concat([algo_selection, new_row_df], ignore_index=True)

    if test_log:
      # preprocessor will determine what preprocessing to perform on what columns
      preprocessor = build_preprocessor(X)

      # Logistic Regression
      logx_train, logx_test, logy_train, logy_test = train_test_split(X, y, test_size=0.2, random_state=0)

      # preprocess data and select DecisionTree
      pipe = Pipeline([('preprocessor', preprocessor), ('Logistic Regression', LogisticRegression())])
      pipe.fit(logx_train, logy_train)
      print(f"Logistic {dataset.metadata["uci_id"]} accuracy is {pipe.score(logx_test, logy_test)}")

    if test_nn:
      # preprocessor will determine what preprocessing to perform on what columns
      preprocessor = build_preprocessor(X)

      # Neural Network
      nnx_train, nnx_test, nny_train, nny_test = train_test_split(X, y, test_size=0.2, random_state=0)

      # preprocess data and select DecisionTree
      pipe = Pipeline([('preprocessor', preprocessor), ('MLP', MLPClassifier(random_state=1, max_iter=500))])
      pipe.fit(nnx_train, nny_train)
      print(f"NN {dataset.metadata["uci_id"]} accuracy is {pipe.score(nnx_test, nny_test)}")

    if test_tree:
      # Decision Tree

      # preprocessor will determine what preprocessing to perform on what columns
      preprocessor = build_preprocessor(X)

      # preprocess data and select DecisionTree
      pipe = Pipeline([('preprocessor', preprocessor), ('Tree', DecisionTreeClassifier(random_state=0))])

      cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

      results = cross_validate(
          pipe,
          X,
          y,
          cv=cv,
          scoring=["accuracy", "f1_macro"],
          return_train_score=False
      )

      accuracy_scores = results["test_accuracy"]
      f1_scores = results["test_f1_macro"]
      mean_accuracy = np.mean(accuracy_scores)
      std_accuracy = np.std(accuracy_scores)

      mean_f1 = np.mean(f1_scores)
      std_f1 = np.std(f1_scores)

      confidence_score = mean_f1 - std_f1

      print(f"Tree {dataset.metadata["uci_id"]} accuracy scores: {accuracy_scores}")
      print(f"Tree {dataset.metadata["uci_id"]} F1 scores: {f1_scores}")
      print(f"Tree {dataset.metadata["uci_id"]} mean accuracy: {mean_accuracy:.4f}")
      print(f"Tree {dataset.metadata["uci_id"]} std accuracy: {std_accuracy:.4f}")
      print(f"Tree {dataset.metadata["uci_id"]} mean F1: {mean_f1:.4f}")
      print(f"Tree {dataset.metadata["uci_id"]} std F1: {std_f1:.4f}")
      print(f"Tree {dataset.metadata["uci_id"]} confidence score (mean F1 - std F1): {confidence_score:.4f}")


    # Determine Recommended Algorithm

Tree 53 accuracy scores: [0.96666667 0.96666667 0.86666667 0.96666667 0.9       ]
Tree 53 F1 scores: [0.96658312 0.96658312 0.86111111 0.96658312 0.89974937]
Tree 53 mean accuracy: 0.9333
Tree 53 std accuracy: 0.0422
Tree 53 mean F1: 0.9321
Tree 53 std F1: 0.0439
Tree 53 confidence score (mean F1 - std F1): 0.8882
Tree 45 accuracy scores: [0.45901639 0.47540984 0.45901639 0.45       0.56666667]
Tree 45 F1 scores: [0.29671306 0.28957265 0.24461514 0.20454545 0.35760483]
Tree 45 mean accuracy: 0.4820
Tree 45 std accuracy: 0.0431
Tree 45 mean F1: 0.2786
Tree 45 std F1: 0.0516
Tree 45 confidence score (mean F1 - std F1): 0.2270
Tree 186 accuracy scores: [0.57923077 0.59615385 0.60046189 0.61508853 0.57967667]
Tree 186 F1 scores: [0.32702174 0.33360401 0.34348363 0.37493743 0.34147818]
Tree 186 mean accuracy: 0.5941
Tree 186 std accuracy: 0.0135
Tree 186 mean F1: 0.3441
Tree 186 std F1: 0.0165
Tree 186 confidence score (mean F1 - std F1): 0.3276
Tree 17 accuracy scores: [0.88596491 0.938596

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Tree 320 accuracy scores: [0.12307692 0.11538462 0.17692308 0.13076923 0.17829457]
Tree 320 F1 scores: [0.10105007 0.0735127  0.11656864 0.06691164 0.18223043]
Tree 320 mean accuracy: 0.1449
Tree 320 std accuracy: 0.0272
Tree 320 mean F1: 0.1081
Tree 320 std F1: 0.0413
Tree 320 confidence score (mean F1 - std F1): 0.0668
Tree 320 accuracy scores: [0.1        0.13076923 0.17692308 0.12307692 0.14728682]
Tree 320 F1 scores: [0.08685992 0.10950942 0.11933812 0.06725324 0.11217712]
Tree 320 mean accuracy: 0.1356
Tree 320 std accuracy: 0.0257
Tree 320 mean F1: 0.0990
Tree 320 std F1: 0.0193
Tree 320 confidence score (mean F1 - std F1): 0.0798


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Tree 109 accuracy scores: [0.91666667 0.83333333 0.97222222 0.97142857 0.94285714]
Tree 109 F1 scores: [0.91931217 0.83722349 0.97432099 0.97011046 0.94368581]
Tree 109 mean accuracy: 0.9273
Tree 109 std accuracy: 0.0513
Tree 109 mean F1: 0.9289
Tree 109 std F1: 0.0500
Tree 109 confidence score (mean F1 - std F1): 0.8790
Tree 19 accuracy scores: [0.97398844 0.97398844 0.9566474  0.97101449 0.95362319]
Tree 19 F1 scores: [0.98065494 0.94721127 0.89095084 0.899596   0.8528449 ]
Tree 19 mean accuracy: 0.9659
Tree 19 std accuracy: 0.0089
Tree 19 mean F1: 0.9143
Tree 19 std F1: 0.0448
Tree 19 confidence score (mean F1 - std F1): 0.8695
Tree 697 accuracy scores: [0.69717514 0.69152542 0.68135593 0.66666667 0.6821267 ]
Tree 697 F1 scores: [0.63894908 0.6374959  0.62133025 0.59304329 0.61765248]
Tree 697 mean accuracy: 0.6838
Tree 697 std accuracy: 0.0104
Tree 697 mean F1: 0.6217
Tree 697 std F1: 0.0166
Tree 697 confidence score (mean F1 - std F1): 0.6051
Tree 350 accuracy scores: [0.727      

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Tree 1 accuracy scores: [0.1937799  0.18899522 0.18682635 0.24431138 0.1988024 ]
Tree 1 F1 scores: [0.118205   0.11952748 0.1021686  0.16397283 0.09135968]
Tree 1 mean accuracy: 0.2025
Tree 1 std accuracy: 0.0213
Tree 1 mean F1: 0.1190
Tree 1 std F1: 0.0248
Tree 1 confidence score (mean F1 - std F1): 0.0943
Tree 891 accuracy scores: [0.79834831 0.79777673 0.79730369 0.7989199  0.79964916]
Tree 891 F1 scores: [0.59394775 0.59626818 0.59351371 0.59725065 0.59814983]
Tree 891 mean accuracy: 0.7984
Tree 891 std accuracy: 0.0008
Tree 891 mean F1: 0.5958
Tree 891 std F1: 0.0018
Tree 891 confidence score (mean F1 - std F1): 0.5940
Tree 94 accuracy scores: [0.90879479 0.91521739 0.91630435 0.91086957 0.91847826]
Tree 94 F1 scores: [0.90451435 0.91160428 0.9123674  0.90732187 0.91413115]
Tree 94 mean accuracy: 0.9139
Tree 94 std accuracy: 0.0036
Tree 94 mean F1: 0.9100
Tree 94 std F1: 0.0035
Tree 94 confidence score (mean F1 - std F1): 0.9065
Tree 468 accuracy scores: [0.86253041 0.86455799 0.8